# Rapport d'évaluation RAGAS — NBA Analyst AI

Ce notebook retrace les évaluations RAGAS successives du pipeline RAG (`ragas_part/evaluate_ragas.py`, dataset de 15 questions, seuil de succès `SCORE_THRESHOLD = 0.7`) ainsi que les améliorations apportées au système entre chaque itération.

## Évolution des scores RAGAS (moyennes globales, 15 questions)

| Étape | faithfulness | context precision | context recall | answer relevancy |
|---|---|---|---|---|
| Prototype initial | 0.392 | 0.122 | 0.060 | 0.365 |
| Après sécurisation de l'outil SQL + schémas Pydantic | 0.594 | 0.089 | 0.163 | 0.624 |
| Meilleur résultat (version actuelle) | 0.504 | 0.317 | 0.344 | 0.939 |

In [ ]:
import pandas as pd

scores = pd.DataFrame(
    [
        {"etape": "Prototype initial", "faithfulness": 0.392, "context_precision": 0.122, "context_recall": 0.060, "answer_relevancy": 0.365},
        {"etape": "Après sécurisation SQL + Pydantic", "faithfulness": 0.594, "context_precision": 0.089, "context_recall": 0.163, "answer_relevancy": 0.624},
        {"etape": "Meilleur résultat (version actuelle)", "faithfulness": 0.504, "context_precision": 0.317, "context_recall": 0.344, "answer_relevancy": 0.939},
    ]
).set_index("etape")
scores

## Améliorations apportées au système RAG

### 1. Environnement et reproductibilité 
- Environnement reproductible géré avec `uv` (`pyproject.toml` + `uv.lock` figés).
- Configuration centralisée dans `utils/config.py`, secrets dans `.env` (non versionné).
- Mise en place du pipeline d'évaluation RAGAS : dataset de questions/réponses attendues (`ragas_dataset.json`), génération des réponses et contextes via le pipeline réel, calcul des métriques.

### 2. Fiabilisation du pipeline avec Pydantic 
- Schémas Pydantic à chaque étape du pipeline (`utils/schemas.py`) : `RawDocument` → `CleanedDocument` → `Chunk` → `EmbeddedChunk` → `SearchResult`, remplaçant des `dict` non typés.
- Validation des entrées utilisateur (`RAGQuery` : longueur, non-vide) avant envoi au LLM.
- Sortie du LLM forcée dans un schéma structuré (`RAGAnswer`) via Pydantic AI, au lieu de texte libre à parser.
- Documents invalides capturés et ignorés (`ValidationError`) sans interrompre le reste de l'indexation.
- Ajout de l'observabilité Pydantic Logfire sur les appels de l'agent.

### 3. Ajustement des hyperparamètres du LLM 
- Les hyperparamètres du LLM n'étaient jusque-là pas explicitement contrôlés : l'agent tournait avec les valeurs par défaut de l'API Mistral (température élevée), ce qui produisait des réponses trop créatives/peu fiables pour un cas d'usage factuel (statistiques NBA).
- Ajout de `LLM_TEMPERATURE`, `LLM_TOP_P`, `LLM_MAX_TOKENS` dans `utils/config.py`, transmis explicitement à l'agent via `model_settings` (`utils/chatbot.py`).
- Température abaissée à `0.2` (quasi déterministe) pour privilégier la fidélité aux contextes récupérés plutôt que la créativité, `top_p = 0.8` et `max_tokens = 800`.

### 4. Sécurisation de l'outil NL→SQL pour les statistiques NBA 
- Chargement des statistiques NBA dans PostgreSQL (`Sql_db/load_excel_to_db.py`).
- Génération de requêtes SQL en langage naturel via LangChain (prompt few-shot + schéma PostgreSQL introspecté).
- Sécurisation de l'outil : requêtes restreintes aux `SELECT` uniquement (`is_safe_select`), limite de lignes forcée (`enforce_row_limit`).
- Erreurs SQL (requête invalide, colonne inexistante, base injoignable) capturées et renvoyées comme message lisible plutôt que de lever une exception.
- Ajout des noms de colonnes dans la réponse SQL pour un contexte plus exploitable par le LLM juge RAGAS.

### 5. Observabilité et logging 
- Centralisation de la configuration du logging dans `utils/observability.py`.
- Logfire configuré de façon conditionnelle (local par défaut, cloud si `LOGFIRE_TOKEN` défini).
- Pont entre le logger standard Python et Logfire (tous les `logging.info/warning/error` remontent automatiquement).
- Runs RAGAS journalisés directement dans Logfire (`ragas_evaluation_run`) au lieu d'un fichier JSON local.

### 6. Organisation, tests et outillage
- Réorganisation du projet en modules dédiés (`ragas_part/`, `Sql_db/`) pour ne pas surcharger `utils/`.
- Makefile pour lancer facilement les scripts (`make index`, `make ragas-refresh`, `make test`, etc.).
- Tests unitaires (`tests/unit`), fonctionnels (`tests/functional`) et d'intégration (`tests/integration`, skippés sans identifiants réels).
- Contexte de l'outil SQL intégré à la génération du dataset RAGAS, pour évaluer aussi les réponses basées sur les statistiques structurées.

## Constats

- `context_precision` et `context_recall` restent les points faibles : les contextes récupérés ne couvrent pas encore correctement les besoins des questions statistiques complexes (agrégations, comparaisons entre équipes/joueurs).
- `answer_relevancy` a fortement progressé (0.365 → 0.939) grâce à la sécurisation de l'outil SQL, aux schémas Pydantic et à l'enrichissement du contexte (noms de colonnes, outil SQL dans le dataset RAGAS).
- Aucun run n'a encore atteint le seuil global (`SCORE_THRESHOLD = 0.7`) sur les quatre métriques simultanément : la prochaine piste d'amélioration porte sur la récupération de contexte (`context_precision`/`context_recall`).